### <span style='color:lightgreen;'>Local Splits Method สำหรับ Image Thresholding</span>

**Local Splits Method** คือการแบ่งภาพออกเป็นส่วนย่อย แล้วหา threshold ของแต่ละส่วนแยกกัน แทนที่จะใช้ threshold ค่าเดียวกับทั้งภาพ

แนวคิดนี้เหมาะกับภาพที่แสงไม่สม่ำเสมอ เช่น บางมุมสว่าง บางมุมมืด เพราะแต่ละส่วนของภาพอาจต้องการ threshold คนละค่า

#### <span style='color:lightblue;'>Workflow ของบทเรียน</span>

1. เตรียม library และฟังก์ชันที่ใช้ซ้ำ
2. อ่านภาพ grayscale
3. ทดลองทำ global threshold ด้วยค่าเฉลี่ยทั้งภาพ
4. แบ่งภาพเป็น 4 ส่วน
5. หา threshold ของแต่ละส่วนจากค่าเฉลี่ยของส่วนนั้น
6. รวมผลลัพธ์กลับเป็นภาพ binary ภาพเดียว
7. เปรียบเทียบผล global threshold กับ local split threshold

1. **<span style='color:lightgreen;'>Import Libraries</span>**

ใช้ `cv2` สำหรับอ่านภาพ, `numpy` สำหรับจัดการ array และ `matplotlib` สำหรับแสดงผลใน notebook

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (6, 4)

2. **<span style='color:lightgreen;'>Helper Functions</span>**

ฟังก์ชันชุดนี้ออกแบบให้สั้นและตรงหน้าที่: แสดงภาพ, แบ่งภาพ 4 ส่วน, threshold ภาพด้วยค่า `T` และรวม block กลับเป็นภาพเดียว

In [ ]:
def show_gray(img, title="Image"):
    plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    plt.title(title)
    plt.axis("off")
    plt.show()


def show_many(images, titles, cols=2, figsize=(10, 6)):
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)

    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[len(images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def split4(img):
    h, w = img.shape
    h_mid, w_mid = h // 2, w // 2

    return [
        ("top-left", img[:h_mid, :w_mid], (slice(0, h_mid), slice(0, w_mid))),
        ("top-right", img[:h_mid, w_mid:], (slice(0, h_mid), slice(w_mid, w))),
        ("bottom-left", img[h_mid:, :w_mid], (slice(h_mid, h), slice(0, w_mid))),
        ("bottom-right", img[h_mid:, w_mid:], (slice(h_mid, h), slice(w_mid, w))),
    ]


def apply_threshold(img, T):
    return np.where(img > T, 255, 0).astype(np.uint8)


def local_mean_threshold(blocks):
    binary_blocks = []
    thresholds = []

    for name, block, region in blocks:
        T = int(round(block.mean()))
        binary_blocks.append((name, apply_threshold(block, T), region))
        thresholds.append((name, T))

    return binary_blocks, thresholds


def merge_blocks(blocks, shape):
    out = np.zeros(shape, dtype=np.uint8)

    for name, block, region in blocks:
        out[region] = block

    return out

3. **<span style='color:lightgreen;'>Read an Image</span>**

ตัวอย่างนี้ใช้ภาพ `gray3.png` จาก folder `images` และอ่านเป็น grayscale เพื่อให้ pixel แต่ละจุดมีค่าความเข้มตั้งแต่ `0` ถึง `255`

In [ ]:
def demo_img(size=512):
    y, x = np.indices((size, size))
    background = 80 + 120 * (x / size) + 35 * (y / size)
    obj = np.zeros((size, size))

    cv2.rectangle(obj, (90, 90), (255, 410), 65, thickness=-1)
    cv2.circle(obj, (350, 250), 95, 70, thickness=-1)

    noise = np.random.default_rng(9).normal(0, 7, (size, size))
    return np.clip(background + obj + noise, 0, 255).astype(np.uint8)


img_path = Path("./images/gray3.png")
img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

if img is None:
    print(f"Image file not found: {img_path}")
    img = demo_img()
else:
    print(f"Image loaded from: {img_path}")

print(f"Image shape: {img.shape}")
print(f"Data type: {img.dtype}")
print(f"Minimum intensity: {img.min()}")
print(f"Maximum intensity: {img.max()}")

In [ ]:
show_gray(img, "Original Grayscale Image")

4. **<span style='color:lightgreen;'>Global Threshold ด้วยค่าเฉลี่ยทั้งภาพ</span>**

วิธี global ใช้ threshold เพียงค่าเดียวกับทุก pixel ในภาพ ตัวอย่างนี้ใช้ค่าเฉลี่ยของภาพเป็น `T`

```text
ถ้า pixel > T  -> 255
ถ้า pixel <= T -> 0
```

ข้อดีคือเข้าใจง่ายและ code สั้น แต่ถ้าภาพมีแสงไม่เท่ากันทั้งภาพ ผลลัพธ์อาจแยกวัตถุได้ไม่ดีในบางบริเวณ

In [ ]:
T_global = int(round(img.mean()))
global_bin = apply_threshold(img, T_global)

print(f"Global threshold from image mean: {T_global}")

show_many(
    [img, global_bin],
    ["Original", f"Global Threshold (T = {T_global})"],
    cols=2,
    figsize=(10, 4),
)

5. **<span style='color:lightgreen;'>Split Image เป็น 4 ส่วน</span>**

เราแบ่งภาพเป็น 4 block ตามตำแหน่ง: บนซ้าย, บนขวา, ล่างซ้าย และล่างขวา แต่ละ block จะถูกนำไปหา threshold ของตัวเอง

In [ ]:
blocks = split4(img)

for name, block, region in blocks:
    print(f"{name:12s} shape: {block.shape}, mean: {block.mean():.2f}")

show_many(
    [block for name, block, region in blocks],
    [name for name, block, region in blocks],
    cols=2,
    figsize=(8, 8),
)

6. **<span style='color:lightgreen;'>Local Threshold ของแต่ละ Block</span>**

ในแต่ละ block เราใช้ค่าเฉลี่ยของ block นั้นเป็น threshold ดังนั้น block ที่มืดและ block ที่สว่างจะได้ค่า `T` ต่างกัน

วิธีนี้ช่วยสื่อแนวคิดสำคัญของ local thresholding ได้ชัดเจน: **ตัดสิน pixel โดยดูบริบทใกล้ ๆ ของ pixel นั้น ไม่ใช่ดูจากภาพทั้งภาพเพียงค่าเดียว**

In [ ]:
binary_blocks, thresholds = local_mean_threshold(blocks)

for name, T in thresholds:
    print(f"{name:12s} local threshold: {T}")

show_many(
    [block for name, block, region in binary_blocks],
    [f"{name} (T = {T})" for (name, T) in thresholds],
    cols=2,
    figsize=(8, 8),
)

7. **<span style='color:lightgreen;'>Merge Blocks กลับเป็นภาพเดียว</span>**

หลังจาก threshold แต่ละ block แล้ว ให้นำ block ทั้ง 4 ส่วนกลับไปวางตำแหน่งเดิม เพื่อสร้าง binary image ผลลัพธ์ของ local split method

In [ ]:
local_bin = merge_blocks(binary_blocks, img.shape)

show_many(
    [img, global_bin, local_bin],
    ["Original", f"Global (T = {T_global})", "Local Split 2 x 2"],
    cols=3,
    figsize=(12, 4),
)

8. **<span style='color:lightgreen;'>เปรียบเทียบ Threshold ที่ใช้</span>**

ตารางสั้น ๆ นี้ช่วยให้นักศึกษาเห็นว่า global method ใช้ค่าเดียว ส่วน local method ใช้ค่าแตกต่างกันตามสภาพแสงของแต่ละบริเวณ

In [ ]:
print(f"{'method':12s} {'region':12s} {'threshold':>10s}")
print("-" * 36)
print(f"{'global':12s} {'whole image':12s} {T_global:10d}")

for name, T in thresholds:
    print(f"{'local':12s} {name:12s} {T:10d}")

#### <span style='color:Tomato;'>สรุปท้ายบท</span>

Local Splits Method เป็นวิธี thresholding แบบง่ายที่แบ่งภาพเป็นส่วนย่อย แล้วให้แต่ละส่วนมี threshold ของตัวเอง

สิ่งที่ควรจำคือ

- global threshold ใช้ค่าเดียวกับทั้งภาพ จึงง่ายแต่ไม่ยืดหยุ่น
- local threshold ใช้ค่าแตกต่างกันตามบริเวณ จึงเหมาะกับภาพที่แสงไม่สม่ำเสมอ
- การแบ่ง block มากขึ้นอาจช่วยให้ละเอียดขึ้น แต่ถ้าแบ่งมากเกินไปอาจเกิดรอยต่อหรือผลลัพธ์ไม่เรียบ
- ในบทนี้ใช้ค่าเฉลี่ยเป็น threshold เพื่อให้เข้าใจแนวคิดก่อน จากนั้นจึงค่อยต่อยอดไปใช้ Otsu หรือ adaptive threshold แบบอื่นได้